# KOHLER AI Bathroom Designer — Data Pipeline

This notebook is the **single data-preparation pipeline** between the trusted primitive dataset and the recommender.

### Design principle
**The dataset is the source of truth. The recommender does not define the dataset schema.**

Pipeline:

**Trustified Primitive -> Cleaning -> Normalization -> Style Classification -> Missing Audit -> KNN Imputation -> Provenance -> Luxury Score -> Validation -> Final Dataset**

### Important boundaries
- Values present in the primitive dataset are treated as source-backed values.
- Values that were previously KNN/fallback-derived are deliberately reset to missing before this pipeline.
- `water_score`, `compact_score`, `footprint`, and final recommendation `score` are **not** created here.
- `luxury_score` is a **project-derived feature**, not a KOHLER-published score.
- Price is **not imputed**. If KOHLER price is unavailable, it remains missing.
- Only three trust flags are used: `price_is_imputed`, `dimension_is_imputed`, `water_is_imputed`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

INPUT = Path("KOHLER_trustified_primitive_dataset_v3.xlsx")
if not INPUT.exists():
    INPUT = Path("/mnt/data/KOHLER_trustified_primitive_dataset_v3.xlsx")

df = pd.read_excel(INPUT)
df.head()


,product_id,name,category,price,width_cm,depth_cm,style,water_usage,water_unit,price_source,dimension_source,water_source
0,K001,Patio™ wall hung toilet with Quiet-Close™ seat...,Toilet,10500.0,36.1,53.5,Traditional,NaN,NaN,KOHLER India product page/catalogue,KOHLER technical/catalogue,Missing — to be imputed by pipeline
1,K002,Span Round wall hung toilet with Quiet-Close™ ...,Toilet,11500.0,36.5,54.0,Minimalist,NaN,NaN,KOHLER India product page/catalogue,KOHLER technical/catalogue,Missing — to be imputed by pipeline
2,K003,Span Square wall hung toilet with Quiet-Close™...,Toilet,12900.0,37.2,54.4,Minimalist,NaN,NaN,KOHLER India product page/catalogue,KOHLER technical/catalogue,Missing — to be imputed by pipeline
3,K004,Reach Eco wall hung toilet with Quiet-Close™ s...,Toilet,17000.0,54.4,37.2,Modern,NaN,NaN,KOHLER India product page/catalogue,KOHLER technical/catalogue,Missing — to be imputed by pipeline
4,K005,Trace™ wall hung toilet with Quiet-Close™ slim...,Toilet,18690.0,37.2,54.4,Minimalist,NaN,NaN,KOHLER India product page/catalogue,KOHLER technical/catalogue,Missing — to be imputed by pipeline


## 1. Cleaning and normalization

In [2]:
# Standardize text fields
text_cols = ["product_id", "name", "category", "style",
             "price_source", "dimension_source", "water_source"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Standardize categories
category_map = {
    "toilets": "Toilet", "toilet": "Toilet",
    "faucets": "Faucet", "faucet": "Faucet",
    "showers": "Shower", "shower": "Shower",
    "vanities": "Vanity", "vanity": "Vanity"
}
df["category"] = df["category"].str.lower().map(category_map).fillna(df["category"])

# Numeric conversion
for col in ["price", "width_cm", "depth_cm", "water_usage"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalize units
df["water_unit"] = df["water_unit"].replace({
    "LPM": "L/min", "lpm": "L/min", "L/minute": "L/min",
    "L/flush": "L/flush", "l/flush": "L/flush"
})

# Water unit is category-defined for the water-consuming fixture groups.
df.loc[df["category"].eq("Toilet") & df["water_usage"].notna(), "water_unit"] = "L/flush"
df.loc[df["category"].isin(["Faucet", "Shower"]) & df["water_usage"].notna(), "water_unit"] = "L/min"

# Vanity does not directly consume water.
vanity = df["category"].eq("Vanity")
df.loc[vanity, ["water_usage", "water_unit"]] = np.nan
df.loc[vanity, "water_source"] = "Not applicable — vanity does not consume water directly"

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        76 non-null     string 
 1   name              76 non-null     string 
 2   category          76 non-null     object 
 3   price             74 non-null     float64
 4   width_cm          45 non-null     float64
 5   depth_cm          45 non-null     float64
 6   style             76 non-null     string 
 7   water_usage       54 non-null     float64
 8   water_unit        54 non-null     object 
 9   price_source      76 non-null     string 
 10  dimension_source  76 non-null     string 
 11  water_source      76 non-null     string 
dtypes: float64(4), object(2), string(6)
memory usage: 7.3+ KB


## 2. Style classification

In [3]:
# Controlled style vocabulary used by the recommender.
STYLE_LABELS = [
    "Modern", "Minimalist", "Japanese Zen", "Traditional",
    "Transitional", "Luxury", "Industrial", "Organic"
]

def classify_style(row):
    text = f"{row['name']} {row['category']}".lower()

    # Strong product/collection cues first
    if any(k in text for k in ["luxe", "statement", "intelligent", "smart toilet", "c3-230"]):
        return "Luxury"
    if any(k in text for k in ["finial", "san raphael", "traditional"]):
        return "Traditional"
    if any(k in text for k in ["foreward", "awaken", "organic", "nature-inspired"]):
        return "Organic"
    if any(k in text for k in ["composed", "transitional"]):
        return "Transitional"
    if any(k in text for k in ["aleo", "prologue", "veil", "components", "span", "evoke", "brazn"]):
        return "Minimalist"
    if any(k in text for k in ["modernlife", "avid", "apt", "vivo", "vive", "fore arc", "fore tri"]):
        return "Modern"

    # Preserve an already supplied controlled label when no stronger cue exists.
    current = str(row.get("style", ""))
    return current if current in STYLE_LABELS else "Modern"

df["style"] = df.apply(classify_style, axis=1)
df["style"].value_counts()


style
Modern         36
Minimalist     25
Luxury          9
Organic         5
Traditional     1
Name: count, dtype: int64

## 3. Missing-value audit

In [4]:
audit = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
audit


,missing_count,missing_pct
product_id,0,0.0
name,0,0.0
category,0,0.0
price,2,2.6
width_cm,31,40.8
depth_cm,31,40.8
style,0,0.0
water_usage,22,28.9
water_unit,22,28.9
price_source,0,0.0


## 4. KNN imputation

**Policy:** KNN within the same category using standardized numeric features.

For each numeric field:
1. Use only rows in the same category where the target value exists.
2. Standardize available numeric features.
3. Find similar products with KNN.
4. Predict the missing target.
5. Set the relevant `is_imputed` flag.

If there are not enough suitable training records, use the **same-category median** as a transparent fallback.

**Price is intentionally excluded from imputation** because an estimated price would be misleading for a budget-constrained recommender.


In [5]:
df["price_is_imputed"] = False
df["dimension_is_imputed"] = False
df["water_is_imputed"] = False

def knn_or_median_fill(frame, target, flag_col, min_train=4, n_neighbors=5):
    numeric_features = [c for c in ["price", "width_cm", "depth_cm", "water_usage"]
                        if c != target]

    for category, group_idx in frame.groupby("category").groups.items():
        idx = list(group_idx)
        train_mask = frame.loc[idx, target].notna()

        if train_mask.sum() < min_train:
            median = frame.loc[idx, target].median()
            if pd.notna(median):
                miss_idx = frame.loc[idx, target].isna()
                frame.loc[frame.loc[idx].index[miss_idx], target] = median
                frame.loc[frame.loc[idx].index[miss_idx], flag_col] = True
            continue

        # Features must be available for both training and prediction.
        usable_features = [
            c for c in numeric_features
            if frame.loc[idx, c].notna().sum() >= min_train
        ]

        miss_idx = frame.loc[idx, target].isna()
        if not miss_idx.any():
            continue

        if not usable_features:
            median = frame.loc[idx, target].median()
            frame.loc[frame.loc[idx].index[miss_idx], target] = median
            frame.loc[frame.loc[idx].index[miss_idx], flag_col] = True
            continue

        train = frame.loc[idx].loc[train_mask, usable_features]
        y = frame.loc[idx].loc[train_mask, target]
        predict_rows = frame.loc[idx].loc[miss_idx, usable_features]

        # KNN needs complete feature vectors.
        complete_train = train.notna().all(axis=1)
        train = train.loc[complete_train]
        y = y.loc[complete_train]

        if len(train) < min_train:
            median = frame.loc[idx, target].median()
            frame.loc[frame.loc[idx].index[miss_idx], target] = median
            frame.loc[frame.loc[idx].index[miss_idx], flag_col] = True
            continue

        scaler = StandardScaler()
        X_train = scaler.fit_transform(train)

        complete_pred = predict_rows.notna().all(axis=1)
        pred_indices = predict_rows.index[complete_pred]

        if len(pred_indices):
            X_pred = scaler.transform(predict_rows.loc[pred_indices])
            k = min(n_neighbors, len(X_train))
            model = KNeighborsRegressor(n_neighbors=k, weights="distance")
            model.fit(X_train, y)
            frame.loc[pred_indices, target] = model.predict(X_pred)
            frame.loc[pred_indices, flag_col] = True

        # Any remaining missing rows use same-category median.
        remaining = frame.loc[idx, target].isna()
        if remaining.any():
            median = frame.loc[idx, target].median()
            if pd.notna(median):
                rem_indices = frame.loc[idx].index[remaining]
                frame.loc[rem_indices, target] = median
                frame.loc[rem_indices, flag_col] = True

    return frame

# Dimensions and water usage can be estimated; price cannot.
df = knn_or_median_fill(df, "width_cm", "dimension_is_imputed")
df = knn_or_median_fill(df, "depth_cm", "dimension_is_imputed")
df = knn_or_median_fill(df, "water_usage", "water_is_imputed")

# Water units follow the category once water usage exists.
df.loc[df["category"].eq("Toilet") & df["water_usage"].notna(), "water_unit"] = "L/flush"
df.loc[df["category"].isin(["Faucet", "Shower"]) & df["water_usage"].notna(), "water_unit"] = "L/min"
df.loc[df["category"].eq("Vanity"), ["water_usage", "water_unit"]] = np.nan

print("Imputed dimensions:", int(df["dimension_is_imputed"].sum()))
print("Imputed water values:", int(df["water_is_imputed"].sum()))
print("Missing prices retained:", int(df["price"].isna().sum()))


Imputed dimensions: 31
Imputed water values: 16
Missing prices retained: 2


/opt/pyvenv/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## 5. Provenance

In [6]:
# Preserve original source fields for trusted values and make imputation provenance explicit.
df.loc[df["dimension_is_imputed"], "dimension_source"] = "KNN / category-median imputation"
df.loc[df["water_is_imputed"], "water_source"] = "KNN / category-median imputation"

# Prices are never imputed.
df.loc[df["price"].isna(), "price_source"] = "Missing — no trusted KOHLER price in primitive dataset"

df[[
    "product_id", "price_source", "dimension_source", "water_source",
    "price_is_imputed", "dimension_is_imputed", "water_is_imputed"
]].head(10)


,product_id,price_source,dimension_source,water_source,price_is_imputed,dimension_is_imputed,water_is_imputed
0,K001,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
1,K002,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
2,K003,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
3,K004,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
4,K005,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
5,K006,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
6,K007,KOHLER India product page/catalogue,KOHLER technical/catalogue,KOHLER product specification,False,False,False
7,K008,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
8,K009,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
9,K010,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True


## 6. Project-derived luxury score

`luxury_score` is **not a KOHLER field**. It is derived for this case study so the recommender can represent a luxury preference.

Method:
- Compare price relative to other products in the same category.
- Add a small premium-product signal for clearly luxury-oriented product names.
- Normalize to a 1–10 range within each category.


In [7]:
luxury_terms = ["luxe", "statement", "intelligent", "smart", "veil", "premium", "electronic", "touchless"]

def luxury_score_for_group(group):
    price = group["price"]
    if price.notna().sum() <= 1:
        base = pd.Series(5.0, index=group.index)
    else:
        lo, hi = price.min(), price.max()
        if hi == lo:
            base = pd.Series(5.0, index=group.index)
        else:
            base = 1 + 9 * (price - lo) / (hi - lo)

    premium = group["name"].str.lower().apply(
        lambda x: min(1.0, sum(term in x for term in luxury_terms) * 0.25)
    )
    score = base.fillna(5.0) + 1.0 * premium
    return score.clip(1, 10).round(2)

df["luxury_score"] = (
    df.groupby("category", group_keys=False)
      .apply(luxury_score_for_group)
      .reset_index(level=0, drop=True)
)

df[["product_id", "category", "price", "luxury_score"]].head(10)


/tmp/ipykernel_1034/4211325904.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(luxury_score_for_group)


,product_id,category,price,luxury_score
0,K001,Toilet,10500.0,1.44
1,K002,Toilet,11500.0,1.48
2,K003,Toilet,12900.0,1.51
3,K004,Toilet,17000.0,1.59
4,K005,Toilet,18690.0,1.67
5,K006,Toilet,23000.0,1.96
6,K007,Toilet,25000.0,2.06
7,K008,Toilet,28000.0,2.59
8,K009,Toilet,32500.0,2.76
9,K010,Toilet,50500.0,3.82


## 7. Validation

Checks:
- Product IDs are unique.
- Required identity fields are complete.
- Dimensions are numeric and positive.
- Water units match fixture category.
- Vanities have no direct water-usage value.
- Price is never silently imputed.
- Only the three `is_imputed` flags are present.
- Dynamic recommender fields such as `water_score`, `compact_score`, `footprint`, and final `score` are absent.


In [8]:
errors = []

if df["product_id"].duplicated().any():
    errors.append("Duplicate product_id values found.")

for col in ["product_id", "name", "category", "style"]:
    if df[col].isna().any():
        errors.append(f"Missing required identity field: {col}")

if (df["width_cm"].dropna() <= 0).any():
    errors.append("Non-positive width found.")
if (df["depth_cm"].dropna() <= 0).any():
    errors.append("Non-positive depth found.")
if (df["water_usage"].dropna() <= 0).any():
    errors.append("Non-positive water usage found.")

bad_toilet_units = df.loc[
    df["category"].eq("Toilet") & df["water_usage"].notna(), "water_unit"
].ne("L/flush").any()
bad_flow_units = df.loc[
    df["category"].isin(["Faucet", "Shower"]) & df["water_usage"].notna(), "water_unit"
].ne("L/min").any()
if bad_toilet_units or bad_flow_units:
    errors.append("Water units do not match category.")

if df.loc[df["category"].eq("Vanity"), "water_usage"].notna().any():
    errors.append("Vanity contains direct water-usage values.")

expected_flags = {"price_is_imputed", "dimension_is_imputed", "water_is_imputed"}
if not expected_flags.issubset(df.columns):
    errors.append("Required imputation flags are missing.")

for dynamic in ["water_score", "compact_score", "footprint", "score"]:
    if dynamic in df.columns:
        errors.append(f"Dynamic recommender field incorrectly present: {dynamic}")

if errors:
    raise ValueError("\n".join(errors))

print("VALIDATION PASSED")
print(f"Products: {len(df)}")
print(f"Categories: {df['category'].value_counts().to_dict()}")
print(f"Missing prices retained: {int(df['price'].isna().sum())}")
print(f"Dimension-imputed rows: {int(df['dimension_is_imputed'].sum())}")
print(f"Water-imputed rows: {int(df['water_is_imputed'].sum())}")


VALIDATION PASSED
Products: 76
Categories: {'Shower': 27, 'Faucet': 26, 'Toilet': 17, 'Vanity': 6}
Missing prices retained: 2
Dimension-imputed rows: 31
Water-imputed rows: 16


## 8. Export final dataset

In [9]:
final_cols = [
    "product_id", "name", "category", "price",
    "width_cm", "depth_cm", "style",
    "water_usage", "water_unit", "luxury_score",
    "price_source", "dimension_source", "water_source",
    "price_is_imputed", "dimension_is_imputed", "water_is_imputed"
]

final_df = df[final_cols].copy()

OUT_XLSX = Path("/mnt/data/KOHLER_AI_Bathroom_Designer_FINAL_DATASET_v3.xlsx")
OUT_CSV = Path("/mnt/data/KOHLER_AI_Bathroom_Designer_FINAL_DATASET_v3.csv")

final_df.to_excel(OUT_XLSX, index=False)
final_df.to_csv(OUT_CSV, index=False)

print("Saved:")
print(OUT_XLSX)
print(OUT_CSV)
final_df.head()


Saved:
/mnt/data/KOHLER_AI_Bathroom_Designer_FINAL_DATASET_v3.xlsx
/mnt/data/KOHLER_AI_Bathroom_Designer_FINAL_DATASET_v3.csv


,product_id,name,category,price,width_cm,depth_cm,style,water_usage,water_unit,luxury_score,price_source,dimension_source,water_source,price_is_imputed,dimension_is_imputed,water_is_imputed
0,K001,Patio™ wall hung toilet with Quiet-Close™ seat...,Toilet,10500.0,36.1,53.5,Traditional,4.0,L/flush,1.44,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
1,K002,Span Round wall hung toilet with Quiet-Close™ ...,Toilet,11500.0,36.5,54.0,Minimalist,4.0,L/flush,1.48,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
2,K003,Span Square wall hung toilet with Quiet-Close™...,Toilet,12900.0,37.2,54.4,Minimalist,4.0,L/flush,1.51,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
3,K004,Reach Eco wall hung toilet with Quiet-Close™ s...,Toilet,17000.0,54.4,37.2,Modern,4.0,L/flush,1.59,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True
4,K005,Trace™ wall hung toilet with Quiet-Close™ slim...,Toilet,18690.0,37.2,54.4,Minimalist,4.0,L/flush,1.67,KOHLER India product page/catalogue,KOHLER technical/catalogue,KNN / category-median imputation,False,False,True


### Final schema

| Group | Columns |
|---|---|
| Identity | `product_id`, `name`, `category` |
| Trusted / prepared attributes | `price`, `width_cm`, `depth_cm`, `style`, `water_usage`, `water_unit` |
| Project-derived static feature | `luxury_score` |
| Provenance | `price_source`, `dimension_source`, `water_source` |
| Trust flags | `price_is_imputed`, `dimension_is_imputed`, `water_is_imputed` |

The recommender should now be rewritten **around this schema**, rather than modifying the dataset to fit the old recommender.
